In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.1 Floating-Point Arithmetic and Numerical Error

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.1",
    title="Floating-Point Arithmetic and Numerical Error",
    blurb="Computers do not do real arithmetic. A rigorous look at the number "
    "system every later notebook actually runs on — machine epsilon, "
    "cancellation, and why energy drifts by 1e-11 and not zero.",
    difficulty="intermediate",
    estimate="80–110 min",
)

## Notebook overview

We have spent (probably) years already computing with **real numbers**: a
mathematically dense continuum, infinitely divisible, and closed under
arithmetic. Yet our computers do not have these mathematical idealisations:
they have instead a finite set of **floating-point numbers**, a grid with gaps
that thins out as we move away from zero. Every `+`, `-`, `*`, `/` we have ever
run silently rounded its exact result back onto that grid. Most of the time we
never notice. This notebook is about the times we must.

The material here is not new physics or new mathematics: it is the
*computational reality* of arithmetic we already trust. We will measure
**machine epsilon**, the size of the gaps; watch **catastrophic cancellation**
destroy the quadratic formula and rebuild it; trace the **U-shaped error
curve** of a numerical derivative (one cannot make it arbitrarily accurate by
shrinking the step); tame **summation error** with compensated addition; and,
finally, explain the number every reader of Volumes I–II has already seen: why
a conserved energy drifts by $\sim 10^{-11}$ and not by zero. That drift is not
a bug; it is the floor this notebook accounts for.

This is the opening notebook of **Volume 0 — Foundations**, and it reaches far
forward: the tolerance arguments here are *why* `ecp.validate.close` compares
with `rtol`/`atol` and never with `==`, and the error floor it establishes sits
under every conservation check in the course.

> **How to read the checks.** Each exercise ends with a `validate` call that
> compares our result to something independent: an exact value, an analytic
> limit, a known relationship. A ✓ is strong evidence we got it right; a ✗ is
> not a verdict but a prompt to *locate the discrepancy*. In a notebook about
> rounding the point is sharper than usual: the right answer is often "equal to
> within a tolerance," never "bit-for-bit equal," so every check here is itself
> a small lesson in choosing that tolerance.

> **Scope.** This is a working review, not a numerical-analysis text. The
> standard references are Goldberg's survey {cite}`goldberg1991` and Higham,
> *Accuracy and Stability of Numerical Algorithms* {cite}`higham2002`; the
> IEEE-754 standard is the primary source.

## Theory in brief

### The IEEE-754 double-precision grid

A 64-bit **double** splits its bits into $1$ sign, $11$ exponent, and $52$
mantissa (fraction) bits. A finite (normal) double is

$$
x = \pm\, (1.b_1 b_2 \cdots b_{52})_2 \times 2^{e},
$$

a $53$-bit significand (the leading $1$ is implicit) times a power of two. The
representable numbers are therefore **not the reals** but a finite grid: within
each *binade* $[2^e, 2^{e+1})$ there are exactly $2^{52}$ equally spaced points,
so the absolute spacing **doubles** every time we cross to the next binade and
the grid is far denser near $0$ than near, say, $10^8$.

The relative spacing of that grid is the **machine epsilon**

```{math}
:label: eq-eps
\varepsilon = 2^{-52} \approx 2.22\times10^{-16},
```

defined as the gap between $1$ and the next larger representable double. It is
the single most important number in this notebook: it sets the *relative*
resolution of every stored quantity.

### The rounding model

Real arithmetic is not closed on the grid, so every operation rounds its exact
result to the nearest representable double. The standard model of this is: for
$\circ \in \{+,-,\times,\div\}$ and the **unit roundoff** $u = \varepsilon/2$,

```{math}
:label: eq-rounding
\mathrm{fl}(x \circ y) = (x \circ y)(1 + \delta), \qquad |\delta| \le u = \tfrac{\varepsilon}{2}.
```

Each individual operation is *correctly rounded*: accurate to half an
ULP (unit in the last place). Trouble comes not from one operation but from how
errors **propagate** and **accumulate**. The headline consequence: $0.1$ has no
finite binary expansion, so it is stored with a small error, and
$0.1 + 0.2 \ne 0.3$ exactly.

### Catastrophic cancellation

Subtracting two nearly equal numbers is the classic amplifier. If
$a \approx b$ are each accurate to a relative $\varepsilon$, their stored
values carry *absolute* errors $\sim \varepsilon|a|$; the difference $a-b$ is
small but inherits those absolute errors, so its **relative** error blows up by
the factor $|a|/|a-b|$. No leading significant digits survive. The textbook
victim is the quadratic formula $x = (-b \pm \sqrt{b^2-4ac})/2a$ when
$b^2 \gg 4ac$: there $\sqrt{b^2-4ac} \approx |b|$, and one of the two roots is
computed as the difference of two nearly equal numbers.

### Truncation versus round-off, and an optimal step

A numerical derivative trades two errors against each other. The forward
difference
$f'(x) \approx \big(f(x+h)-f(x)\big)/h$ has a **truncation** error $\sim
\tfrac{h}{2}|f''|$ (from the Taylor remainder, shrinking with $h$) and a
**round-off** error $\sim \varepsilon|f|/h$ (the cancellation in the numerator,
*growing* as $h$ shrinks). Their sum is U-shaped in $h$ and is minimised at

```{math}
:label: eq-optimal-h
h_\star \sim \sqrt{\varepsilon}\,\approx 1.5\times10^{-8},
```

with a minimum error of order $\sqrt{\varepsilon}$. (Setting the two error
terms equal gives $h_\star$ in two lines; Press et al., *Numerical Recipes*,
§5.7, carry the analysis out in full.) One **cannot** make a
numerical derivative arbitrarily accurate by taking $h \to 0$: past $h_\star$
round-off takes over and the answer gets *worse*. (A central difference does
better: optimum near $\varepsilon^{1/3}$, error $\sim \varepsilon^{2/3}$.)

### Accumulation and conditioning

Summing $N$ terms accumulates rounding: the worst case grows like $O(N\varepsilon)$,
but with errors of random sign the typical growth is a **random walk**,
$O(\sqrt{N}\,\varepsilon)$. That $\sqrt{N}\,\varepsilon$ law is the origin of
the $\sim 10^{-11}$ energy drifts we saw in Volumes I–II, now made
quantitative. Compensated (Kahan) summation and pairwise summation tame it.

Finally, distinguish a bad **algorithm** (cancellation: *fixable* by rewriting)
from a badly **conditioned problem** (*intrinsic*). The relative **condition
number** $\kappa$ of evaluating $f$ at $x$,

```{math}
:label: eq-condnum
\frac{|\Delta f|}{|f|} \approx \kappa\,\frac{|\Delta x|}{|x|}, \qquad
\kappa = \left|\frac{x\,f'(x)}{f(x)}\right|,
```

measures how much a relative input perturbation is amplified in the output.
(One first-order Taylor expansion of $f$ delivers both statements; Higham,
*Accuracy and Stability of Numerical Algorithms*, Ch. 1, develops conditioning
in full.) Since the input is already stored with a relative error
$\sim\varepsilon$, the best achievable relative accuracy is
$\sim \kappa\varepsilon$: *no algorithm can do better*. When $\kappa$ is huge,
the problem itself is the limit.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from decimal import Decimal, getcontext

from ecp import validate
from ecp.animate import show

rng = np.random.default_rng(0)

# Machine epsilon and the unit roundoff, used throughout.
EPS = np.finfo(float).eps  # 2**-52 ≈ 2.22e-16
U = EPS / 2.0  # unit roundoff
getcontext().prec = 50  # high-precision reference arithmetic (stdlib)

print(f"machine epsilon  eps = {EPS:.17g}")
print(f"unit roundoff    u   = {U:.17g}")

## Exercise 1 — The floating-point grid

The theory says doubles form a finite grid with relative spacing
$\varepsilon$, {eq}`eq-eps`, that thins out away from zero. This exercise makes
all three facts tangible by direct experiment, so that every later "why is this
not exactly zero?" has a concrete picture behind it.

1. Show that $0.1 + 0.2 \ne 0.3$, and print the *exact* decimal value stored for
   $0.1$ using `decimal.Decimal`: the binary expansion of $0.1$ does not
   terminate, so it is rounded on input.
2. Find machine epsilon *experimentally*, as the smallest $e=2^{-k}$ with
   $1+e \ne 1$, and confirm it equals $\varepsilon$ from {eq}`eq-eps`
   (`numpy.finfo(float).eps`).
3. Show the grid spacing scales with magnitude: compare `numpy.nextafter` steps
   near $1$ and near $10^8$.

The schematic in {numref}`fig-fp-grid` pictures the non-uniform grid the parts
below probe.

In [ ]:
# (solution hidden on the public site)


### Solution

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    found_eps,
    EPS,
    "experimental machine epsilon matches 2⁻⁵²",
    rtol=1e-12,
)
validate.check(
    gap_near_1e8 > gap_near_1,
    "the grid is coarser near 1e8 than near 1 (spacing scales with magnitude)",
    f"gap(1e8)/gap(1) = {gap_near_1e8 / gap_near_1:.3e}",
)

## Exercise 2 — Catastrophic cancellation: the quadratic formula

This is the centrepiece. The rounding model {eq}`eq-rounding` guarantees each
operation is accurate to half an ULP — yet a *sequence* of operations can throw
away almost every significant digit. The naive quadratic formula
$x = (-b \pm \sqrt{b^2-4ac})/2a$ does exactly this for $b^2 \gg 4ac$: one root
is $(-b + \sqrt{b^2-4ac})/2a$, a subtraction of two nearly equal numbers.

1. For $a=1,\ b=10^8,\ c=1$, compute both roots with the textbook formula
   (`numpy.sqrt` on the discriminant) and show the small root's **residual**
   $a x^2 + b x + c$ is far from zero.
2. Implement the numerically stable form: compute
   $q = -\tfrac12\big(b + \operatorname{sign}(b)\sqrt{b^2-4ac}\big)$ with
   `numpy.sign`, and take $x_1 = q/a,\ x_2 = c/q$ (Vieta's $x_1 x_2 = c/a$ avoids
   the cancellation).
3. Compare the residuals, and confirm the stable small root matches the analytic
   small-root limit $-c/b$. The figure sweeps $b$ to show the naive relative
   error climbing while the stable formula stays at the floor.

In [ ]:
# (solution hidden on the public site)


The figure below sweeps $b$ over many decades. For each $b$ we compute the
small root naively and stably, and the relative error against a
50-digit `Decimal` reference: an independent ground truth.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    small_stable,
    -c / b,
    "the stable formula recovers the small root (≈ -c/b)",
    rtol=1e-9,
)
validate.check(
    abs(res_naive) > 1e-3 and abs(res_stable) < 1e-10,
    "the naive small root has a large residual; the stable one does not",
    f"|res_naive| = {abs(res_naive):.3e},  |res_stable| = {abs(res_stable):.3e}",
)

## Exercise 3 — Loss of significance in a familiar identity

A "new look" at something we have differentiated a hundred times. The function
$g(x) = \dfrac{1 - \cos x}{x^2}$ tends to $\tfrac12$ as $x\to0$ — but evaluated
*naively* near $0$ it is noise, because $1-\cos x$ is a subtraction of two
numbers both $\approx 1$ (cancellation again, per {eq}`eq-rounding`). The cure
is algebra, not a smaller step: using $1-\cos x = 2\sin^2(x/2)$,

$$
\frac{1-\cos x}{x^2} = \frac{1}{2}\left(\frac{\sin(x/2)}{x/2}\right)^2,
$$

which has no cancellation.

1. Implement both forms.
2. On a log-spaced $x$ (`numpy.logspace`), show the naive form disintegrates into
   hash as $x\to0$ while the stable form rides smoothly to $\tfrac12$.
3. Confirm the stable form equals $\tfrac12$ to a tight tolerance at very small
   $x$, where the naive form cannot.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    stable_small,
    0.5 * np.ones_like(x_small),
    "the stable form → 1/2 as x→0",
    atol=1e-10,
)
validate.check(
    not np.allclose(naive_small, 0.5, atol=1e-3),
    "the naive form fails to reach 1/2 at tiny x (it is dominated by round-off)",
    f"naive(x={x_small[0]:.0e}) = {naive_small[0]:.3g}",
)

## Exercise 4 — Numerical differentiation and the error U-curve

Recall the trade-off behind {eq}`eq-optimal-h`: the forward-difference
derivative carries a truncation error $\sim \tfrac{h}{2}|f''|$ that *shrinks*
with $h$ and a round-off error $\sim \varepsilon|f|/h$ that *grows* as $h\to0$.
Their sum is U-shaped, minimised at $h_\star \sim \sqrt{\varepsilon}$. This is
the canonical demonstration that smaller is not always better.

1. For $f=\sin$ at $x=1$ (so $f'=\cos 1$ is the exact target), sweep $h$ over
   many decades (`numpy.logspace`) and plot the forward-difference error: the
   signature U.
2. Confirm the empirical optimum $h_\star$ (`numpy.argmin` of the error) matches
   $\sqrt{\varepsilon}$ from {eq}`eq-optimal-h`.
3. Add the **central** difference and show its optimum sits near
   $\varepsilon^{1/3}$ with a markedly lower minimum error.

This points forward: it is *why* the integrators in
[§1.6](../01-elementary-mechanics/integrators.ipynb) (and every ODE solve
in the course) have a sweet-spot step rather than an arbitrarily small one.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
# The minimum is broad AND jagged (round-off makes the small-h arm noisy), so the
# claim of {eq}`eq-optimal-h` is order-of-magnitude: h* ~ √ε. We therefore test
# agreement in log space (within one decade), not with a tight linear tolerance.
log_decades_off = abs(np.log10(h_star_fwd) - np.log10(np.sqrt(EPS)))
validate.check(
    log_decades_off < 1.0,
    "the forward-difference optimum is h* ~ √ε (within a decade)",
    f"h* = {h_star_fwd:.2e},  √ε = {np.sqrt(EPS):.2e}  ({log_decades_off:.2f} decades off)",
)
validate.check(
    cen_err.min() < fwd_err.min(),
    "the central difference reaches a lower minimum error than the forward difference",
    f"min central {cen_err.min():.2e} < min forward {fwd_err.min():.2e}",
)

## Exercise 5 — Summation error and compensated summation

Adding many numbers accumulates the per-operation rounding of {eq}`eq-rounding`.
Summing $N$ copies of $0.1$ should give $0.1N$; naively it does not, and the
error grows with $N$.

1. Sum $10^6$ copies of $0.1$ sequentially (a plain Python loop) and measure the
   error against the exact value $10^5$.
2. Implement **Kahan (compensated) summation**, which carries a running
   correction for the lost low-order bits, and show it is accurate to the floor.
3. Compare against `numpy.sum`, which already uses **pairwise** summation (it
   splits the array and recurses), so its error grows like $O(\log N)$ rather
   than $O(N)$. The figure sweeps $N$ to show the three growth laws.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    err_naive > 1e-7 and err_kahan < 1e-9,
    "Kahan summation eliminates the naive accumulation error",
    f"naive {err_naive:.2e}  vs  Kahan {err_kahan:.2e}",
)

## Exercise 6 — Why physics energy drifts are ~1e-11, not 0

Here is the payoff that ties this notebook to all of mechanics. Throughout
Volumes I–II we validated integrators by watching a conserved energy stay flat.
It never stayed *exactly* flat, though; it drifted at the $10^{-11}$ level. That
floor is the $\sqrt{N}\,\varepsilon$ accumulation of the rounding model, and we
can isolate it cleanly. Our bare-rotation demo does far less arithmetic per step
than a real adaptive solver, so its floor lands about two orders *below* the
$10^{-11}$ of Volumes I–II — yet it follows the identical $\sqrt{N}\,\varepsilon$
law, which is the point.

Take the simple harmonic oscillator $\ddot x = -\omega^2 x$. Its exact
time-$\Delta t_n$ propagator is a **rotation** in the $(x,\,v/\omega)$ plane,

$$
x_{n+1} = x_n\cos\theta_n + \tfrac{v_n}{\omega}\sin\theta_n, \qquad
v_{n+1} = -\omega x_n\sin\theta_n + v_n\cos\theta_n, \qquad \theta_n=\omega\,\Delta t_n,
$$

which conserves $E = \tfrac12(v^2 + \omega^2 x^2)$ **exactly** in real
arithmetic, for *any* steps $\Delta t_n$: there is *zero* truncation error.
So when we iterate it in floating point, any drift in $E$ is **pure
round-off**. We use *irregular* steps $\Delta t_n$ (exactly what an adaptive
solver like the `DOP853` of Volumes I–II does), which makes the per-step
rounding error random in sign; over $N$ steps it **random-walks** to
$\sim\sqrt{N}\,\varepsilon$ rather than cancelling or growing linearly. (With a
single *fixed* step the rounding bias is systematic and the drift instead grows
$\propto N$; irregular steps are the realistic case.)

1. Iterate the exact rotation propagator for $2\times10^5$ irregular steps (step
   sizes drawn with `numpy.random.default_rng`), recording the relative energy
   error $|E_N - E_0|/E_0$ every thousand steps.
2. Plot the accumulated error against $N$ on log–log axes — the right view for a
   power law, where the walk and its $\sqrt{N}\,\varepsilon$ envelope read as
   parallel lines.
3. Confirm the error stays at the floating-point floor and grows far slower than
   the worst-case $N\varepsilon$.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    final_energy_error < 100 * np.sqrt(N_steps) * EPS,
    "energy error stays at the floating-point accumulation floor (~√N·ε)",
    f"error {final_energy_error:.2e}  at the √N·ε floor {np.sqrt(N_steps) * EPS:.2e}  (checked with ×100 head-room)",
)
validate.check(
    final_energy_error < 0.05 * (N_steps * EPS),
    "the error grows far slower than the worst-case linear N·ε (it is a random walk)",
    f"error {final_energy_error:.2e}  vs  N·ε = {N_steps * EPS:.2e}",
)

## Exercise 7 — Conditioning: when the problem, not the algorithm, limits us (student animation)

Cancellation in Exercises 2–3 was a *fixable* defect of the algorithm: rewrite
the formula and the error vanishes. Some problems are not so kind: a badly
**conditioned** problem amplifies error no matter how cleverly one computes it,
because the input is already stored with a relative error $\sim\varepsilon$ and
{eq}`eq-condnum` multiplies it by $\kappa$. The achievable accuracy is
$\sim\kappa\varepsilon$, full stop.

Take $f(x)=\tan x$ near $x=\pi/2$, where $\kappa(x) = |x f'/f| = |2x/\sin 2x|$
diverges.

1. Compute $\kappa(x)$ analytically, and *measure* the amplification by
   perturbing the input by a small relative $\delta$ and forming
   $(\,|\tan(x(1+\delta))-\tan x|/|\tan x|\,)/\delta$ with `numpy.tan`.
2. **You build the animation**: as $x \to \pi/2$, animate the measured
   amplification climbing along the analytic $\kappa(x)$ curve (and the
   achievable-accuracy floor $\kappa\varepsilon$ rising with it). Use
   `FuncAnimation`, `plt.close(fig)`, then `ecp.animate.show`.
3. Confirm the measured amplification tracks $\kappa$, so the relative error
   indeed tracks $\kappa\varepsilon$.

Because the check tests the *data* (amplification vs. $\kappa$), a ✗ means
"re-examine the condition-number formula or the perturbation measurement,"
never "the animation is wrong."

In [ ]:
# (solution hidden on the public site)


Now build the animation (Part b). The data are ready: `xs`, `kappa`,
`amp_measured`, and the `floor`. Sweep a marker along the curve as $x\to\pi/2$.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    amp_measured,
    kappa,
    "the measured relative-error amplification equals the condition number κ",
    rtol=1e-2,
)

## Exercise 8 — Practical hygiene (synthesis)

A short, rigorous wrap-up of the rules every later notebook leans on.

1. **Never test floats with `==`.** $0.1+0.2 \ne 0.3$, yet they agree to within
   a few $\varepsilon$. Compare with a tolerance: exactly what
   `ecp.validate.close(got, expected, rtol=…, atol=…)` does, and why it exists.
2. **`rtol` vs `atol`.** A check passes when
   $|{\rm got}-{\rm expected}| \le {\tt atol} + {\tt rtol}\,|{\rm expected}|$.
   Use `rtol` for quantities with a natural scale (energies, frequencies) and
   `atol` for quantities that should be near zero (a residual, a drift), where a
   relative test is meaningless.
3. **Reach for more precision only when one must, and know its cost.**
   `math.fsum` sums exactly (at $O(N)$ bookkeeping); `decimal` and `mpmath` give
   arbitrary precision but run orders of magnitude slower: fine for a reference
   value (as in Exercise 2), not for an inner loop.

The check below states rule 1 as code: the two numbers are *not* equal, yet they
are equal to within a tolerance.

In [ ]:
validate.check(
    (0.1 + 0.2 != 0.3) and (abs(0.1 + 0.2 - 0.3) < 1e-15),
    "float equality needs tolerances, not == (0.1+0.2 ≠ 0.3, but within a few ε)",
    f"|0.1+0.2-0.3| = {abs(0.1 + 0.2 - 0.3):.3e}",
)

## Exercise 9 — Showpiece: the fast inverse square root

One last flourish, because it is the most beautiful thing floating-point bit
layout has ever bought. Three-dimensional graphics normalise vectors without
pause: every lighting and reflection calculation needs $\hat{\mathbf v} =
\mathbf v / \lVert\mathbf v\rVert$, which is $1/\sqrt{x}$ with $x = v_x^2 + v_y^2
+ v_z^2$, evaluated millions of times per frame. In the 1990s, before CPUs
carried a fast hardware square root, the *Quake III Arena* source did it with a
fragment that became legendary the instant people read it:

```c
i  = 0x5f3759df - ( i >> 1 );               // what the f***?
y  = y * ( 1.5f - ( 0.5f * x * y * y ) );   // 1st iteration
```

(the bewildered comment is in the original). It returns $1/\sqrt{x}$ to a
fraction of a percent using no division and no square root, only an integer
subtraction, a bit shift, and one multiply-heavy polish step. The entire trick is
the floating-point representation that this whole notebook has been about.

**Why it works (the conceptual centre).** Recall the IEEE-754 layout from the
start of this notebook: a float stores a sign, an exponent, and a mantissa, and
for $x = 2^e(1+m)$ the *integer* read straight off those same bits is, up to
scaling, an affine function of $\log_2 x$. At an exact power of two the
relationship is not approximate but **exact**: reading the float32 bits of
$x = 2^e$ as an integer $I$ gives $I/2^{23} - 127 = e = \log_2 x$ on the nose,
because the mantissa field is zero and only the exponent contributes. Part b)
checks this. Since the bits essentially *are* $\log_2 x$, halving and negating
them computes $-\tfrac12\log_2 x = \log_2\!\big(x^{-1/2}\big)$, which is the bit
pattern of $1/\sqrt{x}$. The magic constant `0x5f3759df` is the additive
correction that best re-centres the affine approximation across the mantissa and
restores the exponent bias.

**The polish.** The bit hack alone is good to about $3.4\%$. One step of Newton's
method applied to $f(y) = 1/y^2 - x$ (whose positive root is $1/\sqrt{x}$) gives
the update $y \leftarrow y\,(1.5 - 0.5\,x\,y^2)$ and drops the error to the famous
$\sim0.17\%$; a second step reaches $\sim10^{-6}$. This is a first sighting of
Newton's method, the subject of [§0.2](root-finding.ipynb), its quadratic
convergence already doing real work.

1. Implement it on `float32`, using `.view(np.int32)` and `.view(np.float32)`
   for the bit reinterpretation, with the magic constant stated explicitly.
2. Verify the bits-are-$\log_2$ identity exactly at $x = 1,2,4,8,16$: the heart
   of the matter.
3. Measure the maximum relative error with the magic step only, with one Newton
   step, and with two, across a range of $x$ ({numref}`fig-fp-fast-rsqrt`).

In [ ]:
# (solution hidden on the public site)


Part b) the centre of the whole trick: at exact powers of two the integer read
off the bits equals $\log_2 x$ with *no* error, so the bit pattern carries the
logarithm the algorithm exploits.

In [ ]:
# (solution hidden on the public site)


Part c) sweep $x$ and measure the relative error against the exact `float64`
value, with zero, one, and two Newton steps.

In [ ]:
# (solution hidden on the public site)


The lesson behind the spectacle: a representation trick, exploited honestly, had
enormous practical impact. Normalising vectors for lighting and reflection is so
ubiquitous in real-time graphics that shaving the cost of $1/\sqrt{x}$ helped make
games like *Quake III* run at all on 1990s hardware. The bits of a float are not
an opaque encoding; they are, quite literally, almost a logarithm.

### Validation 9

In [ ]:
validate.close(
    log2_from_bits,
    log2_true,
    "the integer bit value equals log₂(x) exactly at powers of two (why the trick works)",
    atol=1e-6,
)
validate.close(
    max_rel_1newton,
    0.00175,
    "one Newton step brings the fast inverse sqrt to ~0.17% error",
    atol=5e-4,
)

## Notebook summary

- The floating-point grid and machine epsilon ($\approx2.2\times10^{-16}$ for float64);
  **catastrophic cancellation** in the naive quadratic formula and loss of significance, and
  their stable rewrites.
- The numerical-differentiation error **U-curve** (truncation versus round-off, optimal step
  $h\sim\sqrt\epsilon$); compensated (Kahan) summation; why physics energy drifts sit near
  $10^{-11}$, not zero; conditioning; and the fast inverse square root as a showpiece.

## Outlook

- **The rest of IEEE-754:** subnormals (gradual underflow), $\pm\infty$, signed
  zero, and `NaN`: including why `NaN != NaN` is *by design* (it makes
  "is this a number?" testable as `x != x`).
- **Interval arithmetic**, which carries rigorous error bars through a whole
  computation instead of a single rounded value.
- **`float32` vs `float64`:** the speed/memory/accuracy trade behind GPU physics
  and machine learning, where halving precision doubles throughput.
- **Symbolic computation** (SymPy) as the escape hatch from rounding entirely:
  exact at the cost of speed. We will use it to *derive* equations of motion
  in [§2.1](../02-classical-mechanics/lagrangian-sympy.ipynb) rather than to
  evaluate them.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()